# Matimo Notebook 03: Meta-Tools — Agents That Create Their Own Tools

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tallclub/matimo/blob/feat/colab-quickstart-notebook/docs/notebooks/03_meta_tools.ipynb)

This is the most advanced notebook in the series. You will watch a Matimo agent **define and register its own tools at runtime** — without any human intervention.

### What you'll learn
- What meta-tools are and why they matter
- How Matimo's `draft_tool` command works
- How the policy engine governs tool creation (only approved roles can draft tools)
- End-to-end demo: agent writes a YAML tool definition, saves it, and calls it

### Prerequisites
- Complete Notebook 01 (Quickstart) first
- An Anthropic or OpenAI API key

In [ ]:
# Step 1: Install Matimo
!pip install matimo -q

In [ ]:
# Step 2: Set your API key
import os
from getpass import getpass

provider = input('Provider (anthropic/openai): ').strip().lower()
api_key = getpass(f'Enter your {provider} API key: ')

if provider == 'anthropic':
 os.environ['ANTHROPIC_API_KEY'] = api_key
elif provider == 'openai':
 os.environ['OPENAI_API_KEY'] = api_key
else:
 raise ValueError('Unsupported provider. Use anthropic or openai.')

print(f'API key set for {provider}.')

## Understanding Meta-Tools

In traditional AI frameworks, tools are **static** — a developer writes them, deploys them, and the agent uses them. Matimo breaks this pattern.

With Matimo's meta-tools, an agent can:
1. Identify a capability gap during a task
2. Draft a new YAML tool definition to fill that gap
3. Submit it for approval (or auto-approve in dev mode)
4. Register and use the new tool in the same session

> **This is only possible because Matimo tools are YAML-first.** The agent generates YAML (not code), which is safe, inspectable, and policy-governed.

In [ ]:
# Step 3: Create a dev-mode policy that allows tool drafting
import tempfile, os

policy_yaml = '''
version: '1.0.0'
status: draft
execution:
 type: http
rules:
 - name: allow_draft_tool
 condition: execution.type == 'command' && tool.name == 'draft_tool'
 action: ALLOW
 roles: [admin, developer]
 - name: allow_function_calls
 condition: execution.type == 'function'
 action: ALLOW
 - name: block_prod_tools
 condition: tool.tags contains 'prod'
 action: BLOCK
'''

d = tempfile.mkdtemp()
policy_path = os.path.join(d, 'meta_policy.yaml')
with open(policy_path, 'w') as f:
 f.write(policy_yaml)
print(f'Policy written to: {policy_path}')

In [ ]:
# Step 4: Initialize Matimo with auto_discover=True to load built-in tools
# including the special 'draft_tool' meta-tool
import matimo as Matimo

m = await Matimo.init([provider], auto_discover=True, untrusted_paths=[d])
print(f'Loaded {len(m.tools)} tools')
print('Meta-tools available:', [t for t in m.tools if t.startswith('draft')])

## Live Demo: Agent Creates a Tool

We will ask the agent to create a simple calculator tool. Watch how it:
1. Realises no calculator tool exists
2. Calls `draft_tool` to write a YAML definition
3. Registers the tool automatically
4. Uses the new tool to answer the original question

In [ ]:
# Step 5: Ask the agent a task that requires a tool it doesn't have yet
result = await m.execute(
 'Calculate the compound interest on $10,000 at 7% annual rate for 5 years.',
 context={'environment': 'dev', 'roles': ['developer']}
)

print('=== Agent Output ===')
print(result)

In [ ]:
# Step 6: Inspect the tool the agent drafted
import glob

drafted_tools = glob.glob(os.path.join(d, 'draft_*.yaml'))
if drafted_tools:
 with open(drafted_tools[0]) as f:
 print('=== Drafted Tool YAML ===')
 print(f.read())
else:
 print('No draft tools found — agent may have used built-in arithmetic.')

## What Just Happened?

| Step | Description |
|------|-------------|
| 1 | Agent received a task requiring computation |
| 2 | Agent scanned available tools — none matched |
| 3 | Agent called `draft_tool` to create a YAML tool definition |
| 4 | Policy engine checked: developer role + draft status = ALLOW |
| 5 | New tool was registered in the current session |
| 6 | Agent called the new tool and completed the task |

**This is the core innovation of Matimo.** No other framework lets an agent safely extend its own toolset at runtime using pure configuration.

In [ ]:
# Step 7: Try blocking tool creation with a stricter policy
strict_policy = '''
version: '1.0.0'
status: prod
execution:
 type: http
rules:
 - name: block_draft_tool
 condition: tool.name == 'draft_tool'
 action: BLOCK
'''

strict_path = os.path.join(d, 'strict_policy.yaml')
with open(strict_path, 'w') as f:
 f.write(strict_policy)

m2 = await Matimo.init([provider], auto_discover=True, untrusted_paths=[d])

try:
 await m2.execute('draft_tool', {})
 print('ALLOWED (unexpected!)')
except Exception as e:
 print(f'BLOCKED by strict policy: {e}')

---
## Summary

You just saw Matimo's most powerful feature: **agents that create their own tools at runtime.**
| Feature | Status |
|---------|--------|
| 1. Agent-authored tools | Working |
| 2. Policy-governed tool creation | Working |
| 3. YAML-only (no code execution) | Working |
| 4. Blocked in prod environment | Working |

### What's Next?
- Explore the full [Matimo docs](https://github.com/tallclub/matimo)
- Join the community and contribute your own tools
- GitHub: https://github.com/tallclub/matimo